# 0DTE Iron Condor Backtest

Near-expiry iron condor on SPY / QQQ / IWM.

**Strategy**: Enter an iron condor (long put wing + short put + short call + long call wing) 1–3 days before expiration, exit at or near expiration (DTE 0), capturing rapid time decay.

**Risk management**: Take profit at 50% of credit received; stop loss at 2× credit.

**Data note**: EODHD daily data often lacks expiration-day (DTE=0) rows, so `exit_dte_tolerance=1` is used to allow exit matching at DTE 0 or 1. This ensures more trades complete successfully.

**Prerequisites**: Pre-downloaded data via optopsy-data:
```bash
EODHD_API_KEY=... optopsy-data download SPY        # options
optopsy-data download SPY -s                       # stock OHLCV
optopsy-data download QQQ                          # options (optional)
optopsy-data download QQQ -s                       # stock OHLCV (optional)
optopsy-data download IWM                          # options (optional)
optopsy-data download IWM -s                       # stock OHLCV (optional)
```

In [ ]:
from dotenv import load_dotenv

import os

load_dotenv()

print(os.getenv("OPTOPSY_DATA_DIR"))

from options_strategies.odte_iron_condor import OdteIronCondorConfig
from options_strategies.odte_iron_condor import run_odte_iron_condor
from options_strategies.shared import load_odte_data
from options_strategies.benchmark import BenchmarkConfig
from options_strategies.benchmark import run_benchmark
from options_strategies.shared import load_benchmark_data
from backtest_charts import BacktestReport

## Configuration

In [ ]:
config = OdteIronCondorConfig(
    symbol="SPY",
    capital=100_000.0,
    quantity=1,
    multiplier=100,
    max_positions=1,
    # Multi-symbol portfolio (uncomment when data is downloaded)
    symbols=["SPY"],  # ["SPY", "QQQ", "IWM"]
    # Entry cycle: "daily" | "weekly" | "biweekly" | "monthly"
    entry_cycle="biweekly",
    # Delta targeting: short legs 30Δ, wings 10Δ
    short_put_delta=0.30,
    short_put_delta_min=0.25,
    short_put_delta_max=0.35,
    short_call_delta=0.30,
    short_call_delta_min=0.25,
    short_call_delta_max=0.35,
    long_put_delta=0.10,
    long_put_delta_min=0.05,
    long_put_delta_max=0.15,
    long_call_delta=0.10,
    long_call_delta_min=0.05,
    long_call_delta_max=0.15,
    # Two-week iron condor: enter up to 14 DTE, exit at 7 DTE
    max_entry_dte=14,
    exit_dte=13,
    exit_dte_tolerance=1,
    # Risk management
    take_profit=0.5,
    stop_loss=-2.0,
    # Entry timing (informational with daily data)
    entry_time="15:30",
)
config

## Load Data

In [ ]:
options_df = {}
stock_df = {}

for sym in config.symbols:
    print(f"Loading data for {sym}…")
    opts, stk = load_odte_data(
        sym,
        start_date=config.start_date,
        end_date=config.end_date,
    )
    options_df[sym] = opts
    stock_df[sym] = stk
    print(f"  Options: {len(opts):,} rows")
    print(f"  Stock:   {len(stk):,} rows")

# Load benchmark data (SPY, QQQ, IWM option chains for summary table)
from backtest_charts import BENCHMARK_SYMBOLS

benchmark_options = {}
for sym in BENCHMARK_SYMBOLS:
    try:
        opts, _ = load_odte_data(sym)
        benchmark_options[sym] = opts
        print(f"Benchmark {sym}: {len(opts):,} rows")
    except FileNotFoundError:
        print(f"Benchmark {sym}: no data, skipped")

# Load stock data for equity curve overlay
all_stock = {}
for sym in set(config.symbols) | set(BENCHMARK_SYMBOLS):
    try:
        _, stk = load_odte_data(sym)
        all_stock[sym] = stk
    except FileNotFoundError:
        pass

# Load benchmark strategy data (rolling ATM call per symbol)
bm_options = {}
bm_stock = {}
for sym in BENCHMARK_SYMBOLS:
    try:
        opts, stk = load_benchmark_data(sym)
        bm_options[sym] = opts
        bm_stock[sym] = stk
        print(f"Benchmark strategy {sym}: {len(opts):,} option rows, {len(stk):,} stock rows")
    except FileNotFoundError:
        print(f"Benchmark strategy {sym}: no data, skipped")

## Run Backtest

In [ ]:
print("Running 0DTE iron condor backtest…")
result = run_odte_iron_condor(options_df, stock_df, config)

In [ ]:
# Run benchmark per symbol for individual equity curves and summaries
benchmark_results = {}
for sym in BENCHMARK_SYMBOLS:
    if sym not in bm_options or sym not in bm_stock:
        continue
    print(f"Running rolling ATM call benchmark for {sym}…")
    sym_config = BenchmarkConfig(symbols=[sym])
    sym_result = run_benchmark({sym: bm_options[sym]}, {sym: bm_stock[sym]}, sym_config)
    benchmark_results[sym] = sym_result
    print(f"  {sym} trades: {sym_result.summary.get('total_trades', 0)}  "
          f"P&L: ${sym_result.summary.get('total_pnl', 0):,.2f}")

## Run Benchmark

## Summary

In [ ]:
s = result.summary
print("═══ 0DTE Iron Condor Portfolio Summary ═══")
print(f"  Total trades:    {s.get('total_trades', 0)}")
print(f"  Win rate:        {s.get('win_rate', 0):.1%}")
print(f"  Total P&L:       ${s.get('total_pnl', 0):,.2f}")
print(f"  Max drawdown:    {s.get('max_drawdown', 0):.2%}")
print(f"  Sharpe ratio:    {s.get('sharpe_ratio', 0):.2f}")
print(f"  Sortino ratio:   {s.get('sortino_ratio', 0):.2f}")
print(f"  Profit factor:   {s.get('profit_factor', 0):.2f}")
print(f"  Avg days held:   {s.get('avg_days_in_trade', 0):.1f}")

## Per-Leg Results

In [ ]:
for name, leg in result.leg_results.items():
    ls = leg.summary
    print(f"\n  ── {name} leg ──")
    print(
        f"    Trades: {ls.get('total_trades', 0)}  "
        f"Win rate: {ls.get('win_rate', 0):.1%}  "
        f"P&L: ${ls.get('total_pnl', 0):,.2f}"
    )
    tl = leg.trade_log
    if not tl.empty and "exit_type" in tl.columns:
        for exit_type in sorted(tl["exit_type"].unique()):
            count = (tl["exit_type"] == exit_type).sum()
            print(f"    {exit_type}: {count}")

## Trade Log Sample

In [ ]:
if not result.trade_log.empty:
    result.trade_log.head(10)

## Visualizations

### Create Report

In [ ]:
report = BacktestReport(
    result,
    capital=config.capital,
    benchmark_options=benchmark_options,
    stock_data=all_stock,
    benchmark_results=benchmark_results,
)

### Equity Curve

In [ ]:
report.plot_equity()

### Cumulative P&L by Leg

In [ ]:
report.plot_cum_pnl()

### Per-Trade P&L Distribution

In [ ]:
report.plot_pnl_dist()

### Exit Type Breakdown

In [ ]:
report.plot_exits()

### Strategy Summary

In [ ]:
report.plot_summary()

### Full Dashboard

In [ ]:
report.plot_dashboard()